# ⚛️ RATISS-ONE — Entraînement Souverain de la Bouche sur CPU

**Jonathan Evina · RATISS Labs · Yaoundé, Cameroun 🇨🇲**

> **Architecture Duale :** Tissu Scellé (`.ratiss`, 95k neurones) + Bouche Séquentielle (`.bouche`, Trigramme Backoff 65/35)  
> **Runtime 100% CPU standard :** Zéro GPU requis (ne pas gaspiller de quota T4/A100).  
> **Les 4 Piliers :** Neurochirurgie $O(1)$, Audit déterministe au $10^{-9}$, Épisode vif One-Shot, Résonance avec normalisation de degré $\frac{\text{poids}}{\ln(\text{degré})}$.

In [ ]:
import os, subprocess, sys

# Récupération du dépôt officiel RATISS-ONE si exécuté sur Google Colab
if not os.path.exists('ratiss_v3.py'):
    subprocess.check_call([
        'git', 'clone', '--depth', '1',
        'https://github.com/jonathansearch/RATISS-ONE.git', '.'
    ])

# Exécution de la suite complète des 27 tests unitaires
!python3 -m unittest discover -s tests -p 'test_*.py'

## 1. Voir la bouche naître, sans le cerveau

Huit phrases. Le top-5 du jouet est un plafond, pas une preuve. La marche gloutonne doit être française sur quelques mots, puis boucler.


In [ ]:
!python3 ratiss_v3.py --demo


## 2. Audit du tissu scellé

Tu dois voir l'alphabet. Après « il » : `a, abord, aborda…`. Après « était » : `abattu, absente, absolument`. Le vrai mot suivant n'y est presque jamais. **C'est pour ça que le modèle ne parlait pas.**


In [ ]:
!python3 ratiss_v3.py cerveau/RatissOne.ratiss --audit


## ⚡ Démonstration du Pilier 4 : Résonance d'Onde & Normalisation par le Degré

On allume simultanément les concepts `soleil` et `maree`.  
Grâce à la formule $\frac{\text{poids}}{\ln(\max(e, \text{degré}))}$, les hubs syntaxiques (`de`, `le`) avec 20 000 connexions sont étouffés, et le concept sémantique `lune` émerge au **Rang #2 (Podium Direct)** !

In [ ]:
from ratiss_v3 import TissuScelle

tissu = TissuScelle('cerveau/RatissOne.ratiss')
# Activation de la résonance causale avec normalisation de degré
res = tissu.raisonner_par_resonance(['soleil', 'maree'], seuil_energie=0.01, max_sauts=2, normaliser_degre=True)

print('Top 5 des carrefours émergents :')
for i, c in enumerate(res['carrefours'][:5]):
    print(f"  #{i+1} {c['noeud']:<15} énergie: {c['energie']:.4f} (sauts: {c['sauts']})")


## 3. Ton texte

Dépose un `.txt` (une phrase par ligne, ou de la prose) dans le panneau fichiers, à côté de ce notebook. Puis règle le nom ci-dessous.

Garde un second fichier, **jamais** donné à l'éducation. C'est lui qui a le droit de donner le top-5.

Registres : `conte` (narration), `dialogue` (le fichier doit déjà contenir `_q_` et `_r_` : voir `entrainement/organiser_corpus.py`), `science` (prose). Les vérités courtes passent par la cellule des faits, pas par celle-ci.


In [ ]:
CORPUS = "corpus/prince_jouet.txt"  # remplace par ton fichier
TENU_A_LECART = "corpus/tenu_a_lecart.txt"
REGISTRE = "conte"
TOURS = 3

!python3 pipeline/03_entrainer.py \
  --cerveau cerveau/RatissOne.ratiss \
  --corpus {CORPUS} \
  --registre {REGISTRE} \
  --tours {TOURS} \
  --sortie checkpoints/bouche.bouche


In [ ]:
!python3 pipeline/04_evaluer.py \
  --cerveau cerveau/RatissOne.ratiss \
  --bouche checkpoints/bouche.bouche \
  --texte {TENU_A_LECART} \
  --lam 0.75 --memoire 8


Lis le tableau. `ordre` est le dernier mot seul. `lumiere` allume les 8 derniers. `tissu scellé seul` doit rester mauvais : s'il devient bon, tu mesures la mauvaise mémoire.

Si `lumiere` ne bat pas `ordre`, tu n'ajoutes rien. Tu changes de texte, pas d'architecture.


## 4. Faits gelés, à part

Une vérité par ligne. Force 100. La bouche emprunte le chemin, elle ne le devine pas.


In [ ]:
!python3 ratiss_v3.py cerveau/RatissOne.ratiss \
  --charger checkpoints/bouche.bouche \
  --faits corpus/faits_jouet.txt \
  --registre fait \
  --sauver checkpoints/bouche.bouche

!python3 ratiss_v3.py cerveau/RatissOne.ratiss \
  --charger checkpoints/bouche.bouche \
  "parle moi de la lune"


## 5. Écouter

`--lam 1` ordre seul. `--lam 0` thème seul. `--glouton` montre la boucle. Sans `--glouton`, la bouche pioche.


In [ ]:
!python3 ratiss_v3.py cerveau/RatissOne.ratiss \
  --charger checkpoints/bouche.bouche \
  --lam 0.75 --memoire 8 --glouton \
  "il etait une fois"


## 6. Emporter le fichier

`checkpoints/bouche.bouche` est le produit. Le `.ratiss` n'a pas changé. Les deux voyagent ensemble : c'est le berceau.

Pour un vrai corpus (Gutenberg, Wikipédia, OpenAssistant), suis `docs/ENTRAINEMENT.md`. Même commande. Autre fichier. Autre registre. Même loi.


In [ ]:
!python3 pipeline/05_exporter.py --bouche checkpoints/bouche.bouche --sortie checkpoints/bouche.bouche
!ls -lh checkpoints cerveau/RatissOne.ratiss


## 7. La méthode langue, dix pas

Même loi. L'ordre est tenu par `entrainement/entrainer_langue.py` : prose, faits, épisode vif (une passe, contraste coupé), retouche d'une case, top-5 seulement sur l'écart, audit de la formule.

Le détail des dossiers est `docs/ORGANISATION_LANGUE.md`. `--cerveau` ne fait que hasher le tissu. Le graphe n'est pas chargé. Runtime CPU.


In [ ]:
!python3 entrainement/entrainer_langue.py \
  --education corpus/prince_jouet.txt \
  --ecart corpus/tenu_a_lecart.txt \
  --faits corpus/faits_jouet.txt \
  --episodes corpus/episodes_jouet.txt \
  --retouches corpus/retouches_jouet.txt \
  --registre conte --tours 3 \
  --cerveau cerveau/RatissOne.ratiss \
  --sortie sortie/langue_jouet

print(open("sortie/langue_jouet/ORGANISATION.txt", encoding="utf-8").read())
